Question: Will the flight be delayed

In [21]:
import sys
sys.path.insert(0, '..')
import pandas as pd
from src.analysis import *
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

df = pd.read_csv('../data/flight_data_2024.csv')

# sample 10% of the data for faster processing (adjust as needed based on memory constraints), 
# but be aware that this may affect the model's performance and generalizability
# reset the index after sampling to avoid issues with indexing later on
df = df.sample(frac = 0.1, random_state = 42).reset_index(drop = True) 
df = clean_data(df)
df = add_dep_hour(df)



C:\Users\13475\AppData\Local\Temp\ipykernel_4416\1903901341.py:9: DtypeWarning: Columns (0: cancellation_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/flight_data_2024.csv')


In [22]:
# Create a binary target variable for whether the flight was delayed by more than 15 minutes
df['is_delayed_15'] = (df['arr_delay'] > 15).astype(int)

In [23]:
# Select features
features = ['dep_hour', 'month', 'day_of_week', 'distance', 'op_unique_carrier', 'origin', 'dest']

# Add full date once we have a data set that contains multiple years, 
# but for now we only have one year of data so we can skip it

# Convert text tonumberic features using one-hot encoding
df_encoded = pd.get_dummies(df[features], drop_first=True)

# Model will predict 'is_delayed_15' based on the features in df_encoded

In [24]:
# Split data into training and testing sets
x = df_encoded
y = df['is_delayed_15']

x_train, x_test, y_train, y_test = train_test_split(
    x, 
    y, 
    test_size = 0.02, 
    random_state = 42
)

# Train a Random Forest Classifier
model = RandomForestClassifier(
    n_estimators = 100, # number of decision trees
    random_state = 42, # for reproducibility
    n_jobs = -1, # use all available CPU cores for faster training
)

model.fit(x_train, y_train)

print(classification_report(y_test, model.predict(x_test)))

              precision    recall  f1-score   support

           0       0.82      0.92      0.86     11291
           1       0.36      0.19      0.25      2868

    accuracy                           0.77     14159
   macro avg       0.59      0.55      0.55     14159
weighted avg       0.72      0.77      0.74     14159

